# 🏛️ RAG-пайплайн для базы знаний ГОСТ Р 77.*

Строит локальную RAG-систему поверх репозитория [arasskazov/gost-kb](https://github.com/arasskazov/gost-kb).

**Что делает:**
- Динамически загружает все `full.md` из всех доменов (`lci/`, `eskd/`, `interop/`) через `INDEX.md`
- Строит векторный индекс (ChromaDB + multilingual-e5-large с инструкционными префиксами)
- Кэширует индекс на Google Drive (повторный запуск < 1 мин)
- Отвечает на вопросы через DeepSeek API

**Порядок запуска:** выполняйте ячейки сверху вниз (Shift+Enter).

---
**Время первого запуска:** ~5–10 минут (загрузка модели эмбеддингов ~1.2 GB)  
**Последующие запуски:** ~1–2 минуты (кэш восстанавливается с Drive)

## Шаг 1 — Установка зависимостей

In [ ]:
%%capture
!pip install llama-index llama-index-vector-stores-chroma llama-index-embeddings-huggingface
!pip install llama-index-llms-openai-like chromadb sentence-transformers openai
print('✅ Зависимости установлены')

## Шаг 2 — API-ключ DeepSeek

Добавьте ключ через **Secrets** (значок 🔑 в левой панели Colab):  
`Имя: DEEPSEEK_API_KEY` → вставьте значение → включите доступ для ноутбука.

Ключ **никогда не попадёт в код или историю коммитов**.

In [ ]:
from google.colab import userdata

DEEPSEEK_API_KEY = userdata.get('DEEPSEEK_API_KEY')
if not DEEPSEEK_API_KEY:
    raise ValueError("❌ Секрет DEEPSEEK_API_KEY не найден. Добавьте его в Secrets (🔑).")
print("✅ API-ключ загружен из Secrets")

## Шаг 3 — Подключение Google Drive (кэш индекса)

Индекс ChromaDB сохраняется в `MyDrive/gost-kb-chroma/`.  
При повторном запуске индекс восстанавливается — стандарты не перезагружаются.

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')
CHROMA_PATH = '/content/drive/MyDrive/gost-kb-chroma'
os.makedirs(CHROMA_PATH, exist_ok=True)
print(f"✅ Drive подключён. Путь к кэшу: {CHROMA_PATH}")

## Шаг 4 — Динамическая загрузка всех стандартов из GitHub

Список стандартов считывается из `INDEX.md` — новые ГОСТы подхватываются автоматически.  
Загружаются `full.md` из всех доменов: `lci/`, `eskd/`, `interop/`.

In [ ]:
import re
import requests
from llama_index.core import Document

BASE_URL = 'https://raw.githubusercontent.com/arasskazov/gost-kb/main'

# Читаем INDEX.md и извлекаем все пути к full.md
print("🔄 Читаем INDEX.md...")
idx_resp = requests.get(f'{BASE_URL}/INDEX.md', timeout=15)
idx_resp.raise_for_status()

# Извлекаем пути вида: standards/<domain>/<std>/full.md
full_md_paths = re.findall(r'\(standards/[^)]+/full\.md\)', idx_resp.text)
full_md_paths = [p.strip('()') for p in full_md_paths]
print(f"📋 Найдено стандартов в индексе: {len(full_md_paths)}")

documents = []
failed = []

for rel_path in full_md_paths:
    # Определяем домен и номер стандарта из пути standards/<domain>/<std>/full.md
    parts = rel_path.split('/')
    domain = parts[1] if len(parts) >= 4 else 'unknown'
    std_id = parts[2] if len(parts) >= 4 else rel_path

    # Загружаем full.md
    url_full = f'{BASE_URL}/{rel_path}'
    r = requests.get(url_full, timeout=15)
    if r.status_code == 200 and len(r.text) > 100:
        documents.append(Document(
            text=r.text,
            metadata={
                'standard': std_id,
                'domain': domain,
                'source': url_full,
                'type': 'full'
            }
        ))
        print(f'  ✅ {domain}/{std_id}: {len(r.text):,} символов')
    else:
        failed.append(rel_path)
        print(f'  ❌ {rel_path}: статус {r.status_code}')

    # Загружаем summary.md как дополнительный документ
    url_sum = url_full.replace('/full.md', '/summary.md')
    r2 = requests.get(url_sum, timeout=15)
    if r2.status_code == 200 and len(r2.text) > 50:
        documents.append(Document(
            text=r2.text,
            metadata={
                'standard': std_id,
                'domain': domain,
                'source': url_sum,
                'type': 'summary'
            }
        ))

print(f'\n📚 Загружено документов: {len(documents)}')
if failed:
    print(f'⚠️  Не загружены: {failed}')

## Шаг 5 — Построение векторного индекса

> ⏳ Первый запуск: ~5–8 минут (скачивается модель ~1.2 GB и индексируются документы)  
> Повторный запуск: индекс загружается из Drive мгновенно

Используется `MarkdownNodeParser` для разбивки по заголовкам стандартов  
и инструкционные префиксы `multilingual-e5-large` для точного поиска.

In [ ]:
import chromadb
from llama_index.core import VectorStoreIndex, StorageContext, Settings
from llama_index.core.node_parser import MarkdownNodeParser
from llama_index.vector_stores.chroma import ChromaVectorStore
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

# Модель эмбеддингов с инструкционными префиксами (требуется для e5-large)
print('🔄 Загружаем модель эмбеддингов (multilingual-e5-large)...')
embed_model = HuggingFaceEmbedding(
    model_name='intfloat/multilingual-e5-large',
    query_instruction='query: ',
    text_instruction='passage: ',
    max_length=512
)
Settings.embed_model = embed_model

# Разбивка документов по заголовкам Markdown (сохраняет структуру стандартов)
Settings.node_parser = MarkdownNodeParser()
print('✅ Модель и парсер настроены')

# ChromaDB с постоянным хранилищем на Google Drive
chroma_client = chromadb.PersistentClient(path=CHROMA_PATH)
chroma_collection = chroma_client.get_or_create_collection(
    'gost_standards',
    metadata={'hnsw:space': 'cosine'}
)
vector_store = ChromaVectorStore(chroma_collection=chroma_collection)
storage_context = StorageContext.from_defaults(vector_store=vector_store)

# Проверяем: индекс уже существует?
existing_count = chroma_collection.count()
if existing_count > 0:
    print(f'✅ Индекс восстановлен из кэша: {existing_count} чанков')
    index = VectorStoreIndex.from_vector_store(
        vector_store,
        storage_context=storage_context
    )
else:
    print(f'🔄 Индексируем {len(documents)} документов...')
    index = VectorStoreIndex.from_documents(
        documents,
        storage_context=storage_context,
        show_progress=True
    )
    print(f'\n✅ Индекс построен: {chroma_collection.count()} чанков')

## Шаг 6 — Настройка DeepSeek и поискового движка

In [ ]:
from llama_index.llms.openai_like import OpenAILike
from llama_index.core import Settings

SYSTEM_PROMPT = """Ты — эксперт-нормировщик по стандартам серии ГОСТ Р 77.*
(управление жизненным циклом изделий).

Правила:
1. Отвечай ТОЛЬКО на основе предоставленных фрагментов стандартов.
2. Всегда указывай номер ГОСТ и раздел/пункт (например: ГОСТ Р 77.001, п. 3.4).
3. Если информация в предоставленных фрагментах отсутствует — так и скажи,
   не придумывай нормы.
4. При противоречиях между стандартами — укажи оба варианта с источниками.
5. Используй точные формулировки из текста стандарта."""

llm = OpenAILike(
    model='deepseek-chat',
    api_base='https://api.deepseek.com',
    api_key=DEEPSEEK_API_KEY,
    temperature=0.1,
    max_tokens=2048,
    is_chat_model=True,
    system_prompt=SYSTEM_PROMPT
)
Settings.llm = llm

# Поисковый движок: топ-6 релевантных чанков
query_engine = index.as_query_engine(
    similarity_top_k=6,
    response_mode='compact'
)
print('✅ Поисковый движок готов')

## Шаг 7 — Задаём вопросы

Измените текст в переменной `question` и запустите ячейку.

In [ ]:
def ask(question: str, show_sources: bool = True):
    print(f'\n❓ Вопрос: {question}')
    print('─' * 60)
    response = query_engine.query(question)
    print(f'💬 Ответ:\n{response}')
    if show_sources:
        print('\n📎 Источники:')
        seen = set()
        for node in response.source_nodes:
            std = node.metadata.get('standard', '?')
            domain = node.metadata.get('domain', '?')
            src_type = node.metadata.get('type', '?')
            score = node.score or 0
            key = f'{domain}/{std}/{src_type}'
            if key not in seen:
                print(f'  → {domain}/{std} ({src_type}), релевантность: {score:.3f}')
                seen.add(key)
    return response

# ── Пример 1 ──────────────────────────────────────────────────────────────────
ask('Что такое жизненный цикл изделия и какие стадии он включает?')

In [ ]:
# ── Пример 2 ──────────────────────────────────────────────────────────────────
ask('Какова область применения ГОСТ Р 77.102?')

In [ ]:
# ── Пример 3 ──────────────────────────────────────────────────────────────────
ask('В чём разница между понятиями "изделие" и "продукт" в стандартах серии 77?')

In [ ]:
# ── Ваш вопрос ────────────────────────────────────────────────────────────────
question = 'Введите ваш вопрос здесь'
ask(question)

## Шаг 8 — Интерактивный режим (чат)

Запустите ячейку и вводите вопросы в поле ввода.  
Введите `выход` для остановки.

In [ ]:
print('🤖 ГОСТ-эксперт готов. Задавайте вопросы (введите "выход" для остановки)\n')
while True:
    q = input('Вопрос: ').strip()
    if not q or q.lower() in ('выход', 'exit', 'quit'):
        print('Сессия завершена.')
        break
    ask(q)

---
## 📝 Советы

**Хорошие вопросы для этой базы:**
- *Что означает термин X по ГОСТ 77.001?*
- *Какие требования к документации на стадии разработки?*
- *Какие стандарты серии 77 регулируют вопросы утилизации?*
- *Перечислите ключевые термины из ГОСТ 77.301*
- *Чем отличаются требования ЕСКД от ГОСТ Р 77 для документации изделия?*

**Если индекс устарел (добавлены новые ГОСТы):**  
Удалите папку `MyDrive/gost-kb-chroma/` и перезапустите с Шага 4.

**Сброс кэша вручную:**
```python
import shutil
shutil.rmtree(CHROMA_PATH)
print("Кэш удалён — перезапустите с Шага 5")
```

**Стоимость DeepSeek API:**  
`deepseek-chat` — $0.27/1M токенов (вход) и $1.10/1M (выход).  
Один типичный запрос к этой базе ≈ $0.001–0.003.

**Изменения относительно предыдущей версии:**
- ✅ API-ключ убран из кода → загружается из Colab Secrets (🔑)
- ✅ Динамическая загрузка всех доменов (`lci/`, `eskd/`, `interop/`) через `INDEX.md`
- ✅ `MarkdownNodeParser` — разбивка по заголовкам стандартов
- ✅ Инструкционные префиксы `query:` / `passage:` для `multilingual-e5-large`
- ✅ Метаданные включают поле `domain`
- ✅ Кэш ChromaDB на Google Drive (повторный запуск < 1 мин)
- ✅ Один блок установки зависимостей